# Download Qwen3 Embedding Model

This notebook downloads `Qwen/Qwen3-Embedding-0.6B` into `ABSA/embedding_model/qwen3_embedding_0_6b`.

Use this model for few-shot retrieval and example selection. Model weights are ignored by git.

## 1. Install the downloader

This installs only the Hugging Face Hub client needed to download the model files. The inference environment can be installed separately on Vast.

In [1]:
%pip install -U huggingface_hub

Note: you may need to restart the kernel to use updated packages.


## 2. Resolve paths

The code below finds the ABSA folder and sets the local embedding model directory.

In [2]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, *cwd.parents]
ABSA_DIR = next(path for path in candidates if (path / "dataset" / "train.json").exists())
EMBEDDING_MODEL_DIR = ABSA_DIR / "embedding_model" / "qwen3_embedding_0_6b"
EMBEDDING_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("ABSA_DIR:", ABSA_DIR)
print("EMBEDDING_MODEL_DIR:", EMBEDDING_MODEL_DIR)

ABSA_DIR: /workspace/Aspect-Based-Sentiment-Analysis-with-LLMs/distribucio-codi-i-dades-12-03-26/ABSA
EMBEDDING_MODEL_DIR: /workspace/Aspect-Based-Sentiment-Analysis-with-LLMs/distribucio-codi-i-dades-12-03-26/ABSA/embedding_model/qwen3_embedding_0_6b


## 3. Optional login

`Qwen/Qwen3-Embedding-0.6B` is public. If Hugging Face asks for authentication or you hit rate limits, uncomment and run this cell.

In [3]:
# from huggingface_hub import notebook_login
# notebook_login()

## 4. Download the embedding model

Run this on the machine where retrieval will be built. It can be the local machine or the Vast instance.

In [4]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=EMBEDDING_MODEL_DIR,
)

print("Downloaded", MODEL_ID, "to", EMBEDDING_MODEL_DIR)

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 12 files: 100%|██████████| 12/12 [00:11<00:00,  1.06it/s]

Downloaded Qwen/Qwen3-Embedding-0.6B to /workspace/Aspect-Based-Sentiment-Analysis-with-LLMs/distribucio-codi-i-dades-12-03-26/ABSA/embedding_model/qwen3_embedding_0_6b


## 5. Verify files

This checks that the local folder has the minimum files expected for a Hugging Face model snapshot.

In [5]:
required_files = ["config.json", "tokenizer_config.json", "tokenizer.json"]
missing = [name for name in required_files if not (EMBEDDING_MODEL_DIR / name).exists()]
assert not missing, f"Missing files: {missing}"

print("Embedding model folder looks valid.")
for path in sorted(EMBEDDING_MODEL_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{path.name:45s} {size_mb:8.1f} MB")

Embedding model folder looks valid.
.gitattributes                                     0.0 MB
README.md                                          0.0 MB
config.json                                        0.0 MB
config_sentence_transformers.json                  0.0 MB
generation_config.json                             0.0 MB
merges.txt                                         1.6 MB
model.safetensors                               1136.4 MB
modules.json                                       0.0 MB
tokenizer.json                                    10.9 MB
tokenizer_config.json                              0.0 MB
vocab.json                                         2.6 MB


## 6. Optional tokenizer sanity check

This does not load the full embedding model weights. It only verifies that the tokenizer can be loaded from the local folder.

In [6]:
%pip install -U transformers safetensors

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_DIR, trust_remote_code=True)
sample = tokenizer("Represent this restaurant review for retrieving similar ABSA examples: great food and slow service")

print("Tokenizer:", tokenizer.__class__.__name__)
print("Sample token count:", len(sample["input_ids"]))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 67.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 80.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.2/801.2 kB 75.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [transformers] [transformers]
Note: you may need to restart the kernel to use updated packages.


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Tokenizer: Qwen2Tokenizer
Sample token count: 17


## 7. Expected use

The future few-shot retrieval code should load embeddings from:

```python
EMBEDDING_MODEL_DIR = ABSA_DIR / "embedding_model" / "qwen3_embedding_0_6b"
```

The generator model remains in:

```python
MODEL_DIR = ABSA_DIR / "model"
```